# Regulations.gov — Webpage Comment Text with 6 API Keys

Downloads only the written comment text shown on Regulations.gov comment pages for `ICEB-2025-0001-0001`. No PDFs or attachment text are downloaded. The notebook rotates across up to 6 API keys, respects `429` / `Retry-After`, and checkpoints progress so you can resume.

In [26]:
import html
import os
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import requests


## 1. Settings

Use environment variables, or uncomment the local placeholder list and paste your six keys there on your own machine.

In [27]:
DOCUMENT_ID = "ICEB-2025-0001-0001"

API_KEYS = [
    "zgfiEGwgj1pnI4dzErbPhCGNknJjPQ4WFlpQh62R",
    "fHswWCgqaAPksyRaJ8lp5AJncUb0tlDLPNn9XTB4",
    "COwIGgYsIp9jkMQAWLGfijmFfiaaSDXUxus0Wcq2",
    "W8gVxXDZiZZBO47EAqCHKvkYaeHTGYjEIJ2mBs9g",
    "TZbljDFSP1hcZJoNaNltw9Kh0jTlr6hTCkEq1b9t",
    "wIqdBiq6a1es3g2NZAXfgjhpqqkEk3GAS7RIeUIv",
]

# Or paste them locally:
# API_KEYS = ["KEY_1", "KEY_2", "KEY_3", "KEY_4", "KEY_5", "KEY_6"]

API_KEYS = [k for k in API_KEYS if k]
if not API_KEYS:
    raise ValueError("No API keys found. Add at least one key before continuing.")

print(f"Loaded {len(API_KEYS)} API key(s).")

PAGE_SIZE = 250
PAGES_PER_BATCH = 20
CHECKPOINT_EVERY = 100
REQUEST_DELAY = 0.10

ID_CHECKPOINT_FILE = Path(f"{DOCUMENT_ID}_comment_ids.csv")
TEXT_CHECKPOINT_FILE = Path(f"{DOCUMENT_ID}_webpage_text_checkpoint.csv")
OUTPUT_FILE = Path(f"{DOCUMENT_ID}_webpage_comment_text.csv")


Loaded 6 API key(s).


## 2. API-key rotation and cooldown handling

In [28]:
key_available_at = {i: 0 for i in range(len(API_KEYS))}
next_key_index = 0

def get_available_key():
    global next_key_index
    while True:
        now = time.time()
        for _ in range(len(API_KEYS)):
            i = next_key_index
            next_key_index = (next_key_index + 1) % len(API_KEYS)
            if key_available_at[i] <= now:
                return i, API_KEYS[i]
        earliest = min(key_available_at.values())
        wait_seconds = max(1, earliest - now)
        print(f"All {len(API_KEYS)} API keys are cooling down. Waiting {wait_seconds:.0f} seconds...")
        time.sleep(wait_seconds)

def regulations_get(url, params=None, max_retries=50):
    params = {} if params is None else params
    attempts = 0
    while attempts < max_retries:
        key_index, api_key = get_available_key()
        request_params = params.copy()
        request_params["api_key"] = api_key
        try:
            r = requests.get(url, params=request_params, timeout=60)
            if r.status_code == 429:
                try:
                    retry_after = int(r.headers.get("Retry-After", "60"))
                except (TypeError, ValueError):
                    retry_after = 60
                key_available_at[key_index] = time.time() + retry_after
                print(f"Key {key_index + 1} rate limited. Cooldown: {retry_after}s. Trying another key...")
                attempts += 1
                continue
            if r.status_code in (500, 502, 503, 504):
                wait = min(60, 2 ** min(attempts, 6))
                print(f"Server error {r.status_code}. Waiting {wait}s...")
                time.sleep(wait)
                attempts += 1
                continue
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            wait = min(60, 2 ** min(attempts, 6))
            print(f"Request failed: {exc}. Retrying in {wait}s...")
            time.sleep(wait)
            attempts += 1
    raise RuntimeError(f"Request failed after {max_retries} attempts: {url}")


## 3. Test one known comment

In [32]:
TEST_COMMENT_ID = "ICEB-2025-0001-3802"
r = regulations_get(f"https://api.regulations.gov/v4/comments/{TEST_COMMENT_ID}")
attrs = r.json()["data"]["attributes"]
raw = attrs.get("comment")
clean = html.unescape(raw).strip() if isinstance(raw, str) else raw
print("COMMENT TEXT:\n")
print(clean)


COMMENT TEXT:

I am a Supply Chain Manager working in U.S. manufacturing. I want to comment on two provisions of the NPRM: the elimination of “duration of status” with a four-year cap (pp.118–121), and the prohibition on F-1 students pursuing a second master’s degree or changing majors at the same level (p.138).<br/><br/>1. Supply Chains Need Cross-Disciplinary Talent<br/>Running a factory and its supply chain today requires more than just engineering knowledge. We need people who understand operations, data, finance, and logistics. Some of the best hires I’ve managed started in other fields—business or accounting—and retrained into supply chain or industrial engineering through second master’s programs (p.138). That flexibility is exactly what helps us solve bottlenecks and keep production moving. Taking away this retraining option makes it harder to find the talent we need.<br/><br/>2. Four-Year Cap Doesn’t Match Industry Timelines<br/>On pp.118–121, DHS proposes to limit student adm

## 4. Get the document object ID and total comment count

In [34]:
r = regulations_get(f"https://api.regulations.gov/v4/documents/{DOCUMENT_ID}")
object_id = r.json()["data"]["attributes"]["objectId"]
print("Object ID:", object_id)

comments_url = "https://api.regulations.gov/v4/comments"
r = regulations_get(comments_url, params={"filter[commentOnId]": object_id, "page[size]": 5})
TOTAL_COMMENTS = r.json().get("meta", {}).get("totalElements")
print("Total comments:", TOTAL_COMMENTS)


Object ID: 09000064b8f21281
Total comments: 21923


## 5. Collect all comment IDs

This handles collections larger than 5,000 records by continuing with `lastModifiedDate`.

In [8]:
def utc_to_eastern_filter(timestamp):
    dt = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
    return dt.astimezone(ZoneInfo("America/New_York")).strftime("%Y-%m-%d %H:%M:%S")

if ID_CHECKPOINT_FILE.exists():
    id_df = pd.read_csv(ID_CHECKPOINT_FILE)
    comment_ids = id_df["comment_id"].astype(str).tolist()
    print(f"Loaded {len(comment_ids):,} comment IDs from checkpoint.")
else:
    all_comments = {}
    cursor_date = None
    previous_cursor = None
    batch = 1
    while True:
        print(f"\n--- ID Batch {batch} ---")
        batch_last = None
        reached_end = False
        for page in range(1, PAGES_PER_BATCH + 1):
            params = {
                "filter[commentOnId]": object_id,
                "page[size]": PAGE_SIZE,
                "page[number]": page,
                "sort": "lastModifiedDate,documentId",
            }
            if cursor_date is not None:
                params["filter[lastModifiedDate][ge]"] = cursor_date
            r = regulations_get(comments_url, params=params)
            records = r.json().get("data", [])
            if not records:
                reached_end = True
                break
            batch_last = records[-1]
            for record in records:
                all_comments[record["id"]] = record
            print(f"Page {page:>2}: {len(records):>3} | unique IDs: {len(all_comments):,}")
            if TOTAL_COMMENTS is not None and len(all_comments) >= TOTAL_COMMENTS:
                reached_end = True
                break
            if len(records) < PAGE_SIZE:
                reached_end = True
                break
            time.sleep(REQUEST_DELAY)
        if reached_end:
            break
        last_modified = batch_last.get("attributes", {}).get("lastModifiedDate")
        if not last_modified:
            raise RuntimeError("Missing lastModifiedDate for pagination")
        previous_cursor = cursor_date
        cursor_date = utc_to_eastern_filter(last_modified)
        if cursor_date == previous_cursor:
            raise RuntimeError("Pagination cursor did not advance")
        batch += 1
    comment_ids = list(all_comments.keys())
    pd.DataFrame({"comment_id": comment_ids}).to_csv(ID_CHECKPOINT_FILE, index=False)
    print(f"Saved {len(comment_ids):,} comment IDs.")



--- ID Batch 1 ---
Page  1: 250 | unique IDs: 250
Page  2: 250 | unique IDs: 500
Page  3: 250 | unique IDs: 750
Page  4: 250 | unique IDs: 1,000
Page  5: 250 | unique IDs: 1,250
Page  6: 250 | unique IDs: 1,500
Page  7: 250 | unique IDs: 1,750
Page  8: 250 | unique IDs: 2,000
Page  9: 250 | unique IDs: 2,250
Page 10: 250 | unique IDs: 2,500
Page 11: 250 | unique IDs: 2,750
Page 12: 250 | unique IDs: 3,000
Page 13: 250 | unique IDs: 3,250
Page 14: 250 | unique IDs: 3,500
Page 15: 250 | unique IDs: 3,750
Page 16: 250 | unique IDs: 4,000
Page 17: 250 | unique IDs: 4,250
Page 18: 250 | unique IDs: 4,500
Page 19: 250 | unique IDs: 4,750
Page 20: 250 | unique IDs: 5,000

--- ID Batch 2 ---
Page  1: 250 | unique IDs: 5,249
Page  2: 250 | unique IDs: 5,499
Page  3: 250 | unique IDs: 5,749
Page  4: 250 | unique IDs: 5,999
Page  5: 250 | unique IDs: 6,249
Page  6: 250 | unique IDs: 6,499
Page  7: 250 | unique IDs: 6,749
Page  8: 250 | unique IDs: 6,999
Page  9: 250 | unique IDs: 7,249
Page 10: 

## 6. Download only the webpage comment text

No attachment parameter is used. Progress is saved every 100 comments.

In [9]:
def get_comment_text(comment_id):
    r = regulations_get(f"https://api.regulations.gov/v4/comments/{comment_id}")
    attrs = r.json()["data"].get("attributes", {})
    raw_comment = attrs.get("comment")
    clean_comment = html.unescape(raw_comment).strip() if isinstance(raw_comment, str) else raw_comment
    return {
        "comment_id": comment_id,
        "comment_url": f"https://www.regulations.gov/comment/{comment_id}",
        "title": attrs.get("title"),
        "posted_date": attrs.get("postedDate"),
        "comment": clean_comment,
    }

if TEXT_CHECKPOINT_FILE.exists():
    existing_df = pd.read_csv(TEXT_CHECKPOINT_FILE)
    rows = existing_df.to_dict("records")
    completed_ids = set(existing_df["comment_id"].astype(str))
    print(f"Resuming from {len(completed_ids):,} saved comments.")
else:
    rows = []
    completed_ids = set()

remaining_ids = [cid for cid in comment_ids if cid not in completed_ids]
print("Comments remaining:", f"{len(remaining_ids):,}")

for i, comment_id in enumerate(remaining_ids, start=1):
    try:
        rows.append(get_comment_text(comment_id))
    except Exception as exc:
        print(f"Failed {comment_id}: {exc}")
        continue
    if i % CHECKPOINT_EVERY == 0:
        checkpoint_df = pd.DataFrame(rows).drop_duplicates("comment_id", keep="last")
        checkpoint_df.to_csv(TEXT_CHECKPOINT_FILE, index=False)
        print(f"Saved {len(checkpoint_df):,} comments")
    time.sleep(REQUEST_DELAY)

text_df = pd.DataFrame(rows).drop_duplicates("comment_id", keep="last")
text_df.to_csv(TEXT_CHECKPOINT_FILE, index=False)
print(f"Detailed comment records saved: {len(text_df):,}")


Comments remaining: 21,923
Saved 100 comments
Saved 200 comments
Saved 300 comments
Saved 400 comments
Saved 500 comments
Saved 600 comments
Saved 700 comments
Saved 800 comments
Saved 900 comments
Saved 1,000 comments
Saved 1,100 comments
Saved 1,200 comments
Saved 1,300 comments
Saved 1,400 comments
Saved 1,500 comments
Saved 1,600 comments
Saved 1,700 comments
Saved 1,800 comments
Saved 1,900 comments
Saved 2,000 comments
Saved 2,100 comments
Saved 2,200 comments
Saved 2,300 comments
Saved 2,400 comments
Saved 2,500 comments
Saved 2,600 comments
Saved 2,700 comments
Saved 2,800 comments
Saved 2,900 comments
Saved 3,000 comments
Saved 3,100 comments
Request failed: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)). Retrying in 1s...
Saved 3,200 comments
Saved 3,300 comments
Saved 3,400 comments
Saved 3,500 comments
Saved 3,600 comments
Saved 3,700 comments
Saved 3,800 comments
Saved 3,900 comments


KeyboardInterrupt: 

## 7. Keep only comments that actually contain written webpage text

In [35]:
import pandas as pd

# Load your saved comments file
text_df = pd.read_csv("comments.csv")

has_text = text_df["comment"].fillna("").astype(str).str.strip().ne("")

webpage_text_df = text_df.loc[has_text].copy()

print("Comment pages checked:", f"{len(text_df):,}")
print("Pages with written text:", f"{len(webpage_text_df):,}")
print("Pages without written text:", f"{(~has_text).sum():,}")

Comment pages checked: 10,000
Pages with written text: 9,999
Pages without written text: 1


## 8. Verify the example comment

In [37]:
import pandas as pd

# Load the CSV
webpage_text_df = pd.read_csv("comments.csv")

# Comment you want to check
TEST_COMMENT_ID = "ICEB-2025-0001-3802"

# Make sure IDs are treated as strings
webpage_text_df["comment_id"] = webpage_text_df["comment_id"].astype(str).str.strip()

# Find the comment
example = webpage_text_df[
    webpage_text_df["comment_id"] == TEST_COMMENT_ID
]

if not example.empty:
    print("Comment found!\n")
    print(example.iloc[0]["comment"])
else:
    print("Example comment was not found in comments.csv.")

Comment found!

I am a Supply Chain Manager working in U.S. manufacturing. I want to comment on two provisions of the NPRM: the elimination of “duration of status” with a four-year cap (pp.118–121), and the prohibition on F-1 students pursuing a second master’s degree or changing majors at the same level (p.138).<br/><br/>1. Supply Chains Need Cross-Disciplinary Talent<br/>Running a factory and its supply chain today requires more than just engineering knowledge. We need people who understand operations, data, finance, and logistics. Some of the best hires I’ve managed started in other fields—business or accounting—and retrained into supply chain or industrial engineering through second master’s programs (p.138). That flexibility is exactly what helps us solve bottlenecks and keep production moving. Taking away this retraining option makes it harder to find the talent we need.<br/><br/>2. Four-Year Cap Doesn’t Match Industry Timelines<br/>On pp.118–121, DHS proposes to limit student ad